# Simulation of PSI relaxation kinetics with mutants

In [ ]:
%matplotlib inline

# Import packages and functions
import modelbase
import numpy as np
import pandas as pd
import os
import importlib
import sys
import re
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, PercentFormatter
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm, CenteredNorm, SymLogNorm, Normalize
import matplotlib.colors as colors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as mpatches

from PIL import Image

from scipy.signal import find_peaks, savgol_filter
from scipy.stats import iqr
from scipy.integrate import simpson
from scipy.optimize import minimize

from modelbase.ode import Model, Simulator, mca
from modelbase.ode import ratelaws as rl
from modelbase.ode import ratefunctions as rf

from concurrent.futures import ProcessPoolExecutor
from functools import partial
from pathlib import Path
from sympy import Matrix, lambdify, linsolve, symbols
from warnings import warn
from os import listdir
from os.path import join
from functools import reduce
from operator import mul

# Helper functions
sys.path.append("../Code")
import functions as fnc
import calculate_parameters_restruct as prm
import functions_light_absorption as lip

# Import model functions
from get_current_model import get_model
from module_update_phycobilisomes import add_OCP

idx = pd.IndexSlice


from functions_custom_steady_state_simulator import simulate_to_steady_state_custom, _find_steady_state, get_response_coefficients, get_response_coefficients_array, get_response_coefficients_df, calculate_ss_Q_red, get_steadystate_y0
from function_residuals import residual_normalisation
from functions_fluorescence_simulation import make_lights, make_adjusted_lights, create_protocol_NPQ, create_protocol_NPQ_short, create_protocol_noNPQ

## Simulate CET overexpression with FR light

In [ ]:
models = {}

# Default model
m0, y0 = get_model(check_consistency=False, verbose=False, get_y0=True)
models["default"] = m0

# Increased CET
for cet_factor in [2,5,10,50,100]:
    mCET = get_model(check_consistency=False, verbose=False, get_y0=False)
    mCET.update_parameter("vNQ_max", mCET.get_parameter("vNQ_max") * cet_factor)
    models[f"{cet_factor}xCET"] = mCET


In [ ]:
# Define the lights
continuous_light = lip.light_gaussianLED(670, 20) # FR
MT_light = lip.light_gaussianLED(440, 2000) # MT: Blue
# after_light = lip.light_gaussianLED(630, 20)
after_light = lip.light_gaussianLED(670, 20)

# Create the protocol
protocol_FR = fnc.create_protocol_const(
    continuous_light, 0.1, None
)

protocol_FR = fnc.create_protocol_const(
   MT_light , 0.01, protocol_FR
)

protocol_FR = fnc.create_protocol_const(
    after_light, 1, protocol_FR
)

# Simulate and plot
fig,ax= plt.subplots(figsize=(10,7))

for model_name, m in models.items():
    # Get the simulator
    y0_ss = get_steadystate_y0(m, y0, continuous_light)
    s = Simulator(m)
    s.initialise(y0_ss)

    s = fnc.simulate_protocol(s, protocol_FR, retry_unsuccessful=True)
    # Plot
    P700 = s.get_full_results_df().loc[:,"Y2"] / m.get_parameter("PSItot")

    ax.plot(P700, label=model_name)

ax.set_ylabel("Fraction of P700$^+$ [rel.]")
ax.set_xlabel("Time [s]")
ax.legend(title="Model")
fnc.add_lightbar(s, ax, 1000, color="mono", remove_pulses=False, scale="log", size=0.06, time_offset=0, )

## Model Flv2/4 using electrons from 1) CET or 2) the PQ-pool

In [ ]:
from module_update_NQ import vNQ_MM

In [ ]:
def fraction(total, frac, fac):
    return total * frac * fac

def add_flv24_CET(m:Model, flv_fraction, cet_factor=1):
    """Add the Flv2/4 reaction using a fraction of the NDH reaction

    Args:
        m (Model): _description_
        flv_fraction (_type_): _description_
        cet_factor (int, optional): _description_. Defaults to 1.
    """
    m.add_parameters({
        "flv_fraction":flv_fraction,
        "cet_fraction":1-flv_fraction,
        "cet_factor":cet_factor,
    })

    # Add the reduced rate constant of NDH
    m.add_derived_parameter(
        parameter_name="vNQ_max_NDH",
        function=fraction,
        parameters=["vNQ_max", "cet_fraction", "cet_factor"]
    )

    # Add the fractional rate constant of Flv2/4
    m.add_derived_parameter(
        parameter_name="vNQ_max_Flv",
        function=fraction,
        parameters=["vNQ_max", "flv_fraction", "cet_factor"]
    )

    m.update_reaction_from_args(
        rate_name="vNQ",
        function=vNQ_MM,
        stoichiometry={
            "Fd_ox": 2,
            "Q_ox": -1,
            "Hi": 1 / m.get_parameter("bHi"),
            "Ho": -3 / m.get_parameter("bHo"),
        },
        args=["Q_ox", "Fd_red", "vNQ_max_NDH", "KMNQ_Qox", "KMNQ_Fdred"],
        reversible=False,
    )

    # Assumption: using the same kinetics as NDH but not utilizing PQ
    m.add_reaction_from_args(
        rate_name="vFlv2/4",
        function=vNQ_MM,
        stoichiometry={
            "Fd_ox": 2,
            "O2": -0.5,
            "Ho": -2 / m.get_parameter("bHo"),
        },
        args=["Q_ox", "Fd_red", "vNQ_max_Flv", "KMNQ_Qox", "KMNQ_Fdred"],
        reversible=False,
    )
    return m



def add_flv24_PQ(m:Model, k_flv24):
    """Add the Flv2/4 reactions using mass action kinetics from PQ

    Args:
        m (Model): _description_
        k_flv24 (_type_): _description_
    """
    m.add_parameter("k_flv24", k_flv24)  

    m.add_reaction(
        rate_name="vFlv2/4",
        function=rf.mass_action_3,
        stoichiometry={
            "Q_ox": 2,
            "O2": -1,
            "Ho": -4 / m.get_parameter("bHo"),
            "Hi": 4 / m.get_parameter("bHi"),
        },
        modifiers=["Q_red"],  # , "Q_ox", "Keq_vbd"
        dynamic_variables=["O2", "Q_red", "Ho"],  # , "Q_ox", "Keq_vbd"
        parameters=["k_flv24"],
        reversible=False,
    )
    return m

In [ ]:
models_cet = {}

# Default model
m0, y0 = get_model(check_consistency=False, verbose=False, get_y0=True)
models_cet["default"] = m0

# Add flv24 from CET
cet_factor=1
for flv_fraction in np.linspace(0.1,0.9,5):
    mCET = get_model(check_consistency=False, verbose=False, get_y0=False)
    mCET = add_flv24_CET(mCET, flv_fraction, cet_factor)
    models_cet[f"{flv_fraction:.2f}/CET,{cet_factor}xCET"] = mCET

models_pq = {}

# Default model
m0, y0 = get_model(check_consistency=False, verbose=False, get_y0=True)
models_pq["default"] = m0

# Add flv24 from PQ
for kflv24 in np.linspace(0.1, 10, 5):
    mflv = get_model(check_consistency=False, verbose=False, get_y0=False)
    mflv = add_flv24_PQ(mflv, kflv24)
    models_pq[f"kflv={kflv24:.3f}"] = mflv

In [ ]:
# Define the lights
continuous_light = lip.light_gaussianLED(450, 444) # FR
MT_light = lip.light_gaussianLED(450, 2000) # MT: Blue
# after_light = lip.light_gaussianLED(630, 20)
after_light = lip.light_gaussianLED(450, 444)

# Create the protocol
protocol_BL = fnc.create_protocol_const(
    continuous_light, 1, None
)

protocol_BL = fnc.create_protocol_const(
   MT_light , 0.4, protocol_BL
)

protocol_BL = fnc.create_protocol_const(
    after_light, 6.5, protocol_BL
)

# Simulate and plot
fig,axes= plt.subplots(2,1,figsize=(10,12), sharex=True)

for models, ax in zip([models_cet, models_pq], axes.flatten()):
    for model_name, m in models.items():
        # Get the simulator
        y0_ss = get_steadystate_y0(m, y0, continuous_light)
        s = Simulator(m)
        s.initialise(y0_ss)

        s = fnc.simulate_protocol(s, protocol_BL, retry_unsuccessful=True)
        # Plot
        P700 = s.get_full_results_df().loc[:,"Y2"] / m.get_parameter("PSItot")

        ax.plot(P700, label=model_name)

        ax.legend(title="Model")
        ax.set_ylabel("Fraction of P700$^+$ [rel.]")
ax.set_xlabel("Time [s]")
fnc.add_lightbar(s, ax, 3000, color="mono", remove_pulses=False, scale="linear", size=0.06, time_offset=0, )

In [ ]:
# Define the lights
continuous_light = lip.light_gaussianLED(450, 20) # FR
MT_light = lip.light_gaussianLED(450, 300) # MT: Blue
# after_light = lip.light_gaussianLED(630, 20)
after_light = lip.light_gaussianLED(450, 20)

# Create the protocol
protocol_BL = fnc.create_protocol_const(
    continuous_light, 1, None
)

protocol_BL = fnc.create_protocol_const(
   MT_light , 0.4, protocol_BL
)

protocol_BL = fnc.create_protocol_const(
    after_light, 6.5, protocol_BL
)

# Simulate and plot
fig,axes= plt.subplots(2,1,figsize=(10,12), sharex=True)

for models, ax in zip([models_cet, models_pq], axes.flatten()):
    for model_name, m in models.items():
        # Get the simulator
        y0_ss = get_steadystate_y0(m, y0, continuous_light)
        s = Simulator(m)
        s.initialise(y0_ss)

        s = fnc.simulate_protocol(s, protocol_BL, retry_unsuccessful=True)
        # Plot
        P700 = s.get_full_results_df().loc[:,"Y2"] / m.get_parameter("PSItot")

        ax.plot(P700, label=model_name)

        ax.legend(title="Model")
        ax.set_ylabel("Fraction of P700$^+$ [rel.]")
ax.set_xlabel("Time [s]")
fnc.add_lightbar(s, ax, 1000, color="mono", remove_pulses=False, scale="linear", size=0.06, time_offset=0, )

## View the data

In [ ]:
file_paths = Path("../Code/data/Jens_flv24_20250801").glob("*.xlsx")
dats = []

for file in file_paths:
    # Get the genotype
    geno = file.name.split(" ")[1]
    dat = pd.read_excel(file, index_col=0)

    # Drop the NA columns
    dat = dat.dropna(how="all", axis=1)

    # Create the new column headers
    # Get the compounds per column
    comps = dat.columns.str.split(".").map(lambda x:x[0]).to_numpy().reshape(-1,1)

    # Get the genotype specific headers
    genos = np.repeat([[geno, x] for x in range((dat.shape[1]+1) // 3)], repeats=3, axis=0)

    _headers = np.concatenate([genos[:,[0]], comps, genos[:,[1]]], axis=1)

    # Set the new column header
    dat.columns = pd.MultiIndex.from_arrays(_headers.T)

    # add to the data files
    dats.append(dat)

flvdat = pd.concat(dats, axis=1).loc[:7.9,:]

In [ ]:
ax = flvdat.T.groupby(level=[0,1]).mean().T.loc[:,idx[:,"P700"]].plot()